In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import re
import random
import math
import time
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{device}")

cuda


In [8]:
# Data cleaning and reading
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_data(filepath, num_examples=30000):
    eng_sentences = []
    rus_sentences = []

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[:num_examples]:
            parts = line.split('\t')
            if len(parts) >= 2:
                eng_sentences.append(clean_text(parts[0]))
                rus_sentences.append(clean_text(parts[1]))

    return eng_sentences, rus_sentences

eng_data, rus_data = load_data(r'C:\Users\srbuh\PycharmProjects\nlp_course_2026_2\Srbuhi Pupuyan\Lab6\rus.txt', 100000)

print(f"Count of sentences: {len(eng_data)}")
print("Example (Eng):", eng_data[100])
print("Example (Rus):", rus_data[100])

Count of sentences: 100000
Example (Eng): he ran
Example (Rus): он бежал


In [9]:
# Vocabulary
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx2word = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.word_freq = Counter()
        self.idx = 4

    def add_sentence(self, sentence):
        for word in sentence.split():
            self.word_freq[word] += 1
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def __len__(self):
        return len(self.word2idx)

# creating vocabs
eng_vocab = Vocabulary("English")
rus_vocab = Vocabulary("Russian")

for eng, rus in zip(eng_data, rus_data):
    eng_vocab.add_sentence(eng)
    rus_vocab.add_sentence(rus)

print(f"Length of english vocab: {len(eng_vocab)}")
print(f"Length of russian vocab: {len(rus_vocab)}")

Length of english vocab: 7216
Length of russian vocab: 20764


In [10]:
# Dataset and Collate function
class TranslationDataset(Dataset):
    def __init__(self, eng_sentences, rus_sentences, eng_vocab, rus_vocab):
        self.eng_sentences = eng_sentences
        self.rus_sentences = rus_sentences
        self.eng_vocab = eng_vocab
        self.rus_vocab = rus_vocab

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, i):
        eng_sentence = self.eng_sentences[i]
        rus_sentence = self.rus_sentences[i]

        eng_tensor = [self.eng_vocab.word2idx.get(w, 3) for w in eng_sentence.split()]

        # Adding <sos> (1) and <eos> (2)
        rus_tensor = [1] + [self.rus_vocab.word2idx.get(w, 3) for w in rus_sentence.split()] + [2]

        return torch.tensor(eng_tensor), torch.tensor(rus_tensor)

def collate_fn(batch):
    eng_batch, rus_batch = [], []
    for eng_item, rus_item in batch:
        eng_batch.append(eng_item)
        rus_batch.append(rus_item)

    # Padding with 0s
    eng_batch = pad_sequence(eng_batch, padding_value=0, batch_first=False)
    rus_batch = pad_sequence(rus_batch, padding_value=0, batch_first=False)

    return eng_batch, rus_batch

# Data splitting (80% train, 20% validation)
eng_train, eng_val, rus_train, rus_val = train_test_split(eng_data, rus_data, test_size=0.2, random_state=42)

train_dataset = TranslationDataset(eng_train, rus_train, eng_vocab, rus_vocab)
val_dataset = TranslationDataset(eng_val, rus_val, eng_vocab, rus_vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

for src, trg in train_loader:
    print("Length of Source (English) tensor (SeqLen, BatchSize):", src.shape)
    print("Length of target (Russian) tensor (SeqLen, BatchSize):", trg.shape)
    break

Length of Source (English) tensor (SeqLen, BatchSize): torch.Size([6, 128])
Length of target (Russian) tensor (SeqLen, BatchSize): torch.Size([9, 128])


In [12]:
# Model Architecture
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.droput = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=2, dropout=dropout_rate)

    def forward(self, src):
        # src: [seq_len, batch_size]
        embedded = self.droput(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=2, dropout=dropout_rate)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0)) # [batch_size, output_dim]
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src = [src_len, batch_size]
        # trg = [trg_len, batch_size]
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        # First input token is <sos>
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            top1 = output.argmax(1)

            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1

        return outputs

In [13]:
# Hyperparameters and model initialization
INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(rus_vocab)
EMB_DIM = 256
HID_DIM = 1024
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

PAD_IDX = rus_vocab.word2idx['<pad>']
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Adam optimizer (lr=0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters:')

The model has 55,741,724 trainable parameters:


In [14]:
# Inference
def translate_sentence(sentence, eng_vocab, rus_vocab, model, device, max_len=50):
    model.eval()

    if isinstance(sentence, str):
        sentence = clean_text(sentence)
        tokens = sentence.split()
    else:
        tokens = sentence

    text_to_indices = [eng_vocab.word2idx.get(token, eng_vocab.word2idx['<unk>']) for token in tokens]

    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(sentence_tensor)

    outputs = [rus_vocab.word2idx['<sos>']]

    for _ in range(max_len):
        previous_word = torch.LongTensor([outputs[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(previous_word, hidden, cell)
            best_guess = output.argmax(1).item()

        outputs.append(best_guess)

        if best_guess == rus_vocab.word2idx['<eos>']:
            break

    translated_sentence = [rus_vocab.idx2word[idx] for idx in outputs]

    return " ".join(translated_sentence[1:-1])

model.load_state_dict(torch.load(r'C:\Users\srbuh\PycharmProjects\nlp_course_2026_2\Srbuhi Pupuyan\Lab6\seq2seq_best_model.pt', weights_only=True))

test_sentences = [
    "i am cold",
    "he is a good boy",
    "how are you",
    "run away"
]

print("--- Testing Translations ---")
for sentence in test_sentences:
    translation = translate_sentence(sentence, eng_vocab, rus_vocab, model, device)
    print(f"English: {sentence}")
    print(f"Russian translation: {translation}")
    print("-" * 30)

--- Testing Translations ---
English: i am cold
Russian translation: я холодно
------------------------------
English: he is a good boy
Russian translation: он хороший мальчик
------------------------------
English: how are you
Russian translation: как вы
------------------------------
English: run away
Russian translation: бегите
------------------------------


In [16]:
# N-grams & Clipped Precision
from collections import Counter

def get_ngrams(tokens, n):
    """
    Given a list of tokens and a value of n,
    return a Counter with the count of every n-gram.
    """
    ngrams = []
    # Loop through tokens and extract slices of length n
    for i in range(len(tokens) - n + 1):
        ngram_slice = tuple(tokens[i : i + n])
        ngrams.append(ngram_slice)

    return Counter(ngrams)

def clipped_precision(hypothesis, reference, n):

    # Get n-grams for both hypothesis (model output) and reference (ground truth)
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)

    # Calculate the total number of n-grams in the hypothesis
    total_hyp_ngrams = sum(hyp_ngrams.values())

    # If the generated sentence is too short and has no n-grams of this length, return 0
    if total_hyp_ngrams == 0:
        return 0.0

    clipped_count = 0
    # Iterate over unique n-grams in the hypothesis
    for ngram, count in hyp_ngrams.items():
        # A word/n-gram can only be credited as many times as it appears in the reference!
        # min() prevents the model from gaming the metric by repeating the same word.
        clipped_count += min(count, ref_ngrams[ngram])

    return clipped_count / total_hyp_ngrams

hyp_text = "the match was postponed because of the snow"
ref_text = "the match was postponed because it was snowing"

# Convert strings to lists of tokens (words)
hyp_tokens = hyp_text.split()
ref_tokens = ref_text.split()

print("Hypothesis:", hyp_tokens)
print("Reference: ", ref_tokens)
print("-" * 40)

#  test Unigrams (n=1)
print("--- (n=1) ---")
print("Hypothesis unigrams:", get_ngrams(hyp_tokens, 1))
print("Reference unigrams: ", get_ngrams(ref_tokens, 1))

# calculate the precision for unigrams
prec_1 = clipped_precision(hyp_tokens, ref_tokens, 1)
print(f"Clipped Precision (n=1): {prec_1:.4f}")
print("-" * 40)

# test Bigrams (n=2)
print("--- BIGRAMS (n=2) ---")
prec_2 = clipped_precision(hyp_tokens, ref_tokens, 2)
print(f"Clipped Precision (n=2): {prec_2:.4f}")

Hypothesis: ['the', 'match', 'was', 'postponed', 'because', 'of', 'the', 'snow']
Reference:  ['the', 'match', 'was', 'postponed', 'because', 'it', 'was', 'snowing']
----------------------------------------
--- (n=1) ---
Hypothesis unigrams: Counter({('the',): 2, ('match',): 1, ('was',): 1, ('postponed',): 1, ('because',): 1, ('of',): 1, ('snow',): 1})
Reference unigrams:  Counter({('was',): 2, ('the',): 1, ('match',): 1, ('postponed',): 1, ('because',): 1, ('it',): 1, ('snowing',): 1})
Clipped Precision (n=1): 0.6250
----------------------------------------
--- BIGRAMS (n=2) ---
Clipped Precision (n=2): 0.5714


In [17]:
# Brevity Penalty & BLEU Score
import math

def brevity_penalty(hypothesis, reference):
    c = len(hypothesis) # Length of the generated translation
    r = len(reference)  # Length of the ground truth

    # If hypothesis length is greater or equal to reference length, no penalty (1.0)
    if c >= r:
        return 1.0
    else:
        # The penalty: e^(1 - r/c)
        return math.exp(1 - (r / c))


def bleu_score(hypothesis, reference):
    # 1. Convert strings to list of words
    hyp_tokens = hypothesis.split()
    ref_tokens = reference.split()

    # 2. Compute clipped precision for n = 1, 2, 3, 4
    precisions = []
    for n in range(1, 5):
        p = clipped_precision(hyp_tokens, ref_tokens, n)
        if p == 0:
            return 0.0
        precisions.append(p)

    # 3. Calculate the Geometric Mean using log space (to avoid numerical underflow)
    log_sum = sum(math.log(p) for p in precisions)
    geometric_mean = math.exp(log_sum / 4.0)

    # 4. Calculate Brevity Penalty
    bp = brevity_penalty(hyp_tokens, ref_tokens)

    # 5. Final BLEU score
    return bp * geometric_mean

# ==========================================
# Sanity check
hyp_text = "the match was postponed because of the snow"
ref_text = "the match was postponed because it was snowing"

final_bleu = bleu_score(hyp_text, ref_text)

print(f"Hypothesis: {hyp_text}")
print(f"Reference:  {ref_text}")
print("-" * 40)
print(f"Final BLEU Score: {final_bleu:.4f}")

Hypothesis: the match was postponed because of the snow
Reference:  the match was postponed because it was snowing
----------------------------------------
Final BLEU Score: 0.5170


In [18]:
# Beam Search Decoding
import torch.nn.functional as F

def beam_search(sentence, eng_vocab, rus_vocab, model, device, beam_width=3, max_len=50):
    model.eval()

    # 1. Prepare the input sentence
    if isinstance(sentence, str):
        sentence = clean_text(sentence)
        tokens = sentence.split()
    else:
        tokens = sentence

    text_to_indices = [eng_vocab.word2idx.get(token, eng_vocab.word2idx['<unk>']) for token in tokens]
    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    # 2. Pass the input through the Encoder to get initial hidden/cell states
    with torch.no_grad():
        hidden, cell = model.encoder(sentence_tensor)

    # 3. Initialize the Beam
    sos_idx = rus_vocab.word2idx['<sos>']
    eos_idx = rus_vocab.word2idx['<eos>']

    # beam lists the "live" candidate sequences
    beam = [ (0.0, [sos_idx], hidden, cell) ]

    # completed will store sequences that have hit the <eos> token
    completed = []

    # 4. Decoding loop
    for _ in range(max_len):
        new_beam = []

        for score, seq, h, c in beam:
            if seq[-1] == eos_idx:
                completed.append((score, seq))
                continue

            # The input for the next step is the last generated word
            previous_word = torch.LongTensor([seq[-1]]).to(device)

            with torch.no_grad():
                output, next_h, next_c = model.decoder(previous_word, h, c)

                # Convert raw output logits into log probabilities
                log_probs = F.log_softmax(output.squeeze(0), dim=0)

                # Find the top `beam_width` words for this specific candidate
                top_log_probs, top_indices = log_probs.topk(beam_width)

                for i in range(beam_width):
                    next_word = top_indices[i].item()
                    log_prob = top_log_probs[i].item()

                    # Accumulate the score
                    new_score = score + log_prob
                    new_seq = seq + [next_word]

                    new_beam.append((new_score, new_seq, next_h, next_c))

        if not new_beam:
            break

        # 5. Prune the beam: Keep only the top `beam_width` candidates overall
        # Sort new_beam by score in descending order
        new_beam.sort(key=lambda x: x[0], reverse=True)
        beam = new_beam[:beam_width]

    # Add any remaining candidates in the beam to completed sequences
    completed.extend([(score, seq) for score, seq, _, _ in beam])

    if not completed:
         completed = [(score, seq) for score, seq, _, _ in beam]

    # 6. Find the absolute best sequence among all completed ones
    completed.sort(key=lambda x: x[0], reverse=True)
    best_score, best_seq = completed[0]

    # Convert token indices back to words (excluding <sos> and <eos>)
    translated_words = []
    for idx in best_seq:
        if idx == sos_idx: continue
        if idx == eos_idx: break
        translated_words.append(rus_vocab.idx2word[idx])

    return " ".join(translated_words)

#  Testing Beam Search vs Greedy
test_sentence = "i am cold"

print(f"Original: {test_sentence}")

# The translate_sentence function from Lab5 is Greedy Decoder
greedy_out = translate_sentence(test_sentence, eng_vocab, rus_vocab, model, device)
print(f"Greedy Output:     {greedy_out}")

#  new Beam Search function
beam_out = beam_search(test_sentence, eng_vocab, rus_vocab, model, device, beam_width=5)
print(f"Beam (k=5) Output: {beam_out}")

Original: i am cold
Greedy Output:     я холодно
Beam (k=5) Output: я холодно


In [19]:
# The Experiment & Final Report
import time

# 1. 100 sentences from the Validation set
NUM_SENTENCES = 100
test_pairs = list(zip(eng_val[:NUM_SENTENCES], rus_val[:NUM_SENTENCES]))

# 2. Define the modes we want to test
modes = [
    {"name": "Greedy", "func": lambda src: translate_sentence(src, eng_vocab, rus_vocab, model, device), "width": None},
    {"name": "Beam (w=3)", "func": lambda src: beam_search(src, eng_vocab, rus_vocab, model, device, beam_width=3), "width": 3},
    {"name": "Beam (w=5)", "func": lambda src: beam_search(src, eng_vocab, rus_vocab, model, device, beam_width=5), "width": 5},
    {"name": "Beam (w=10)", "func": lambda src: beam_search(src, eng_vocab, rus_vocab, model, device, beam_width=10), "width": 10}
]

print(f"Running experiment on {NUM_SENTENCES} sentences...\n")

results_table = []
all_translations = {mode["name"]: [] for mode in modes}

# 3. Run the experiment
for mode in modes:
    name = mode["name"]
    func = mode["func"]

    total_time = 0.0
    total_bleu = 0.0

    # Translate all 100 sentences
    for src, trg in test_pairs:
        start_time = time.time()

        # Translate
        pred = func(src)

        end_time = time.time()

        # Calculate time and BLEU
        time_taken = end_time - start_time
        total_time += time_taken

        # Calculate BLEU-4 for this pair
        score = bleu_score(pred, trg)
        total_bleu += score

        # Save the translation to show examples later
        all_translations[name].append(pred)

    avg_time = total_time / NUM_SENTENCES
    avg_bleu = total_bleu / NUM_SENTENCES

    results_table.append((name, avg_bleu, avg_time))

# 4. Print the results table
print("=== EXPERIMENT RESULTS ===")
print(f"{'Decoding Method':<15} | {'Average BLEU-4':<15} | {'Time per sentence (s)':<25}")
print("-" * 60)
for name, bleu, t in results_table:
    print(f"{name:<15} | {bleu:<15.4f} | {t:<25.4f}")
print("\n")

# 5. 5 Concrete Examples side-by-side
print("=== 5 CONCRETE EXAMPLES (Greedy vs Beam w=5) ===")

example_indices = [5, 20, 45, 70, 95]

for idx in example_indices:
    src, trg = test_pairs[idx]
    greedy_pred = all_translations["Greedy"][idx]
    beam5_pred = all_translations["Beam (w=5)"][idx]

    print(f"Source (EN):    {src}")
    print(f"Reference (RU): {trg}")
    print(f"Greedy Out:     {greedy_pred}")
    print(f"Beam (w=5) Out: {beam5_pred}")
    print("-" * 50)

Running experiment on 100 sentences...

=== EXPERIMENT RESULTS ===
Decoding Method | Average BLEU-4  | Time per sentence (s)    
------------------------------------------------------------
Greedy          | 0.0600          | 0.0132                   
Beam (w=3)      | 0.0600          | 0.0574                   
Beam (w=5)      | 0.0600          | 0.1253                   
Beam (w=10)     | 0.0600          | 0.3021                   


=== 5 CONCRETE EXAMPLES (Greedy vs Beam w=5) ===
Source (EN):    i used to admire tom
Reference (RU): я раньше восхищался томом
Greedy Out:     я раньше уважал тома
Beam (w=5) Out: я раньше уважал тома
--------------------------------------------------
Source (EN):    just dont tell tom
Reference (RU): только не говорите тому
Greedy Out:     только просто не тому
Beam (w=5) Out: просто просто тому тому
--------------------------------------------------
Source (EN):    our tv isnt working
Reference (RU): у нас телевизор не работает
Greedy Out:     наш тел